In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, DateType, TimestampType
import pyspark.sql.functions as F

In [0]:
catalog_name="ecommerce"

In [0]:
df=spark.table(f"{catalog_name}.bronze.brz_order_item")

In [0]:
display(df.limit(5))

In [0]:
# Transformation: Convert 'TWO' -> 2 and cast to integer
df=df.withColumn(
    "quantity",
    F.when(F.col("quantity")=="Two",2).otherwise(F.col("quantity")).cast(IntegerType())
)

# Transformation: Remove any '$' or other symbol from unit _price, keep only numeric 
df=df.withColumn(
    "unit_price",
    F.regexp_replace(F.col("unit_price"),"[$]","").cast("double")
)

# Transformation : Remove '%' from discount_pct and cast to double
df=df.withColumn(
    "discount_pct",
    F.regexp_replace(F.col("discount_pct"),"%","").cast("double")
)
#Transformation: coupon code processing (convert to lower)
df=df.withColumn(
    "coupon_code",
    F.lower(F.col("coupon_code"))
)

# Transformation: channel processing
df=df.withColumn(
    "channel",
    F.when(F.col("channel")=="web",'Website')
    .when(F.col("channel")=="app","Mobile")
    .otherwise(F.col("channel"))
    )

In [0]:
display(df.limit(5))

In [0]:
df.printSchema()

In [0]:
# Transfromation: datatype conversions
# 1) Convert dt (string-> date)
df=df.withColumn(
    "dt",
    F.to_date(F.col("dt"),"yyyy-MM-dd")
)

# 2) convert order_ts (string -> timestamp)
df=df.withColumn(
    "order_ts",
    F.coalesce(
        F.to_timestamp("order_ts","yyyy-MM-dd HH:mm:ss"), # matches 2025-08-01 22:53:52
        F.to_timestamp("order_ts","dd-MM-yyyy HH:mm")     # fallback for 01-08-2025 22:53
    )
)

# 3) convert item_seq (string -> integer)

df=df.withColumn(
    "item_seq",
    F.col("item_seq").cast("int")
)

# 4) convert tax_amount (string -> double, strip non-numeric characters)
df=df.withColumn(
    "tax_amount",
    F.regexp_replace(F.col("tax_amount"),r"[^0-9.\-]","").cast("double")
)

# 5)convert product_id (string -> long)
df=df.withColumn(
    "product_id",
    F.col("product_id").cast("bigint")
)

# Transformation : Add processed time
df=df.withColumn(
    "process_time",
    F.current_timestamp()
)


In [0]:
display(df.limit(5))

In [0]:
#write raw data to the layer (catalog ecommers schema silver, table : slv_brands)
df.write.format("delta").mode("overwrite").option("mergeShema","true").saveAsTable(f"{catalog_name}.silver.slv_order_items")